# 🐦 Twitter/X Scraping — Aksesibilitas Mobile Banking untuk Tunanetra
---

| | |
|---|---|
| **Nama** | Fathan Nabil Rahman |
| **Role** | Data Scientist |
| **Tim** | CC26-PSU325 |
| **Program** | Coding Camp 2026 powered by DBS Foundation |
| **Proyek** | VoiceBank: Voice-First Banking App untuk Tunanetra Indonesia |
| **Tema** | Future-Ready Work & Economy |

---

## 🎯 Tujuan

Notebook ini mengumpulkan **data primer tambahan** dari Twitter/X menggunakan `tweet-harvest` untuk memperkuat validasi problem statement proyek VoiceBank.

Data Play Store sebelumnya sudah dikumpulkan namun output-nya terlalu random dan tidak cukup mewakili suara pengguna tunanetra. Twitter/X dipilih sebagai sumber alternatif karena:
1. Pengguna dengan disabilitas visual cenderung lebih vokal dan terorganisir di Twitter
2. Komunitas difabel & aksesibilitas digital aktif berdiskusi di platform ini
3. Keluhan konkret terhadap bank/e-wallet tertentu lebih mudah ditemukan melalui keyword search

### 🏦 Target Aplikasi

| Kategori | Aplikasi |
|----------|----------|
| Bank Tradisional | BRI (BRImo), BCA, Mandiri (Livin) |
| E-Wallet / Fintech | Dana, GoPay, OVO |

### 📐 Dua Layer Keyword

| Layer | Fokus | Contoh |
|-------|-------|--------|
| **Layer 1 — Aksesibilitas** | Tweet eksplisit soal tunanetra & alat bantu | `tunanetra BCA`, `talkback Dana` |
| **Layer 2 — UX Umum** | Keluhan UI/navigasi yang memperparah hambatan | `ribet BRImo`, `susah GoPay` |

> **Catatan metodologi:** Layer 2 digunakan dengan argumen yang sama seperti EDA Play Store — jika pengguna awam pun mengeluh soal UX, tunanetra menghadapi hambatan yang bersifat absolut karena tidak memiliki workaround visual.

---

## Konfigurasi Auth Token


In [15]:
import os
from dotenv import load_dotenv

# Load file .env
load_dotenv()

TWITTER_AUTH_TOKEN = os.getenv("TWITTER_AUTH_TOKEN")

if not TWITTER_AUTH_TOKEN:
    raise ValueError("❌ TWITTER_AUTH_TOKEN belum diisi!")

print(f"✅ Auth token dimuat: {TWITTER_AUTH_TOKEN[:8]}...{TWITTER_AUTH_TOKEN[-4:]}")

✅ Auth token dimuat: 04ddd12d...d94f


## Definisi Keyword

Keyword dibagi menjadi dua layer sesuai strategi riset:

In [16]:
import pandas as pd

# =============================================================================
# AKSESIBILITAS TUNANETRA PADA APLIKASI KEUANGAN DIGITAL
# Keyword digunakan untuk mencari tweet yang secara eksplisit
# membahas tunanetra, screen reader, dan aksesibilitas
# pada layanan mobile banking dan e-wallet.
# =============================================================================

KEYWORDS_AKSESIBILITAS = [

    # -------------------------------------------------------------------------
    # Identitas pengguna + nama aplikasi
    # -------------------------------------------------------------------------
    ("tunanetra BRI",              "aksesibilitas_bri_tunanetra"),
    ("tunanetra BCA",              "aksesibilitas_bca_tunanetra"),
    ("tunanetra Mandiri",          "aksesibilitas_mandiri_tunanetra"),
    ("tunanetra Dana",             "aksesibilitas_dana_tunanetra"),
    ("tunanetra GoPay",            "aksesibilitas_gopay_tunanetra"),
    ("tunanetra OVO",              "aksesibilitas_ovo_tunanetra"),

    # -------------------------------------------------------------------------
    # Screen reader + aplikasi
    # -------------------------------------------------------------------------
    ("talkback BRI",               "talkback_bri"),
    ("talkback BCA",               "talkback_bca"),
    ("talkback Mandiri",           "talkback_mandiri"),
    ("talkback Dana",              "talkback_dana"),
    ("talkback GoPay",             "talkback_gopay"),
    ("talkback OVO",               "talkback_ovo"),

    ("screen reader mobile banking", "screenreader_mbanking"),
    ("voiceover mobile banking",   "voiceover_mbanking"),

    # -------------------------------------------------------------------------
    # Aksesibilitas digital umum
    # -------------------------------------------------------------------------
    ("aksesibilitas mobile banking", "aksesibilitas_mbanking"),
    ("aksesibilitas aplikasi bank",  "aksesibilitas_appbank"),
    ("aksesibilitas ewallet",        "aksesibilitas_ewallet"),
    ("disabilitas visual bank",      "disabilitas_visual_bank"),
    ("difabel mobile banking",       "difabel_mbanking"),

    # -------------------------------------------------------------------------
    # Keluhan eksplisit terkait hambatan akses
    # -------------------------------------------------------------------------
    ("tunanetra transfer bank",     "tunanetra_transfer"),
    ("tunanetra tidak bisa bayar",  "tunanetra_bayar"),
    ("buta dibantu transfer",       "buta_transfer"),
    ("blind mobile banking",        "blind_mbanking"),
    ("low vision bank app",         "lowvision_bank"),
]

# =============================================================================
# INFORMASI KEYWORD
# =============================================================================

print("✅ Keyword berhasil dimuat!")
print(f"📌 Total keyword aksesibilitas: {len(KEYWORDS_AKSESIBILITAS)}")

print("\n📋 Daftar keyword:")
for i, (keyword, filename) in enumerate(KEYWORDS_AKSESIBILITAS, start=1):
    print(f"{i:02d}. {keyword} -> {filename}.csv")

✅ Keyword berhasil dimuat!
📌 Total keyword aksesibilitas: 24

📋 Daftar keyword:
01. tunanetra BRI -> aksesibilitas_bri_tunanetra.csv
02. tunanetra BCA -> aksesibilitas_bca_tunanetra.csv
03. tunanetra Mandiri -> aksesibilitas_mandiri_tunanetra.csv
04. tunanetra Dana -> aksesibilitas_dana_tunanetra.csv
05. tunanetra GoPay -> aksesibilitas_gopay_tunanetra.csv
06. tunanetra OVO -> aksesibilitas_ovo_tunanetra.csv
07. talkback BRI -> talkback_bri.csv
08. talkback BCA -> talkback_bca.csv
09. talkback Mandiri -> talkback_mandiri.csv
10. talkback Dana -> talkback_dana.csv
11. talkback GoPay -> talkback_gopay.csv
12. talkback OVO -> talkback_ovo.csv
13. screen reader mobile banking -> screenreader_mbanking.csv
14. voiceover mobile banking -> voiceover_mbanking.csv
15. aksesibilitas mobile banking -> aksesibilitas_mbanking.csv
16. aksesibilitas aplikasi bank -> aksesibilitas_appbank.csv
17. aksesibilitas ewallet -> aksesibilitas_ewallet.csv
18. disabilitas visual bank -> disabilitas_visual_bank.c

## Fungsi Scraping Helper

In [17]:
import os
import time
import shutil
import pandas as pd

# =============================================================================
# KONFIGURASI SCRAPING
# =============================================================================

OUTPUT_DIR = "../data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TWEET_LIMIT = 100
TAB = "LATEST"
DELAY_SEC = 5

# =============================================================================
# FUNGSI SCRAPING
# =============================================================================

def scrape_keyword(keyword: str, filename: str, limit: int = TWEET_LIMIT) -> bool:

    tmp_folder = "tweets-data"
    tmp_path = os.path.join(tmp_folder, f"{filename}.csv")
    final_path = os.path.join(OUTPUT_DIR, f"{filename}.csv")

    # -------------------------------------------------------------------------
    # Skip scraping jika dataset sudah tersedia
    # -------------------------------------------------------------------------
    if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
        print(f"  ⏭️ [{filename}] Sudah ada di folder data, skip scraping.")
        return True

    # -------------------------------------------------------------------------
    # Perintah scraping Twitter/X
    # -------------------------------------------------------------------------
    cmd = (
        f'npx tweet-harvest@2.7.1 '
        f'-o "{filename}.csv" '
        f'-s "{keyword}" '
        f'--tab "{TAB}" '
        f'-l {limit} '
        f'--token "{TWITTER_AUTH_TOKEN}"'
    )

    os.system(cmd)

    # -------------------------------------------------------------------------
    # Pindahkan hasil scraping ke folder data
    # -------------------------------------------------------------------------
    if os.path.exists(tmp_path):

        shutil.move(tmp_path, final_path)

        if os.path.getsize(final_path) > 0:
            try:
                df_tmp = pd.read_csv(final_path)
                print(f"  ✅ [{filename}] → {len(df_tmp)} tweets")
                return True

            except pd.errors.EmptyDataError:
                print(f"  ⚠️ [{filename}] → Hasil kosong, tapi file dibuat.")
                return True

        else:
            print(f"  ⚠️ [{filename}] → Tidak ada hasil (0 tweets).")
            return True

    else:
        # Jika file tidak ditemukan, kemungkinan tidak ada tweet
        print(f"  ℹ️ [{filename}] → Scraping selesai (mungkin 0 hasil).")
        return True


print("✅ Fungsi scraping siap digunakan!")

✅ Fungsi scraping siap digunakan!


## Uji Coba (1 Keyword Dulu)

Jalankan ini dulu untuk pastikan koneksi & auth token valid sebelum scraping massal.

In [20]:
print("🔍 Uji coba scraping 1 keyword...")

success = scrape_keyword(
    "tunanetra mobile banking",
    "test_tunanetra_mbanking",
    limit=50
)

if success:

    df_test = pd.read_csv(
        f"{OUTPUT_DIR}/test_tunanetra_mbanking.csv"
    )

    print("\n📋 Preview hasil scraping:")
    
    print(
        df_test[
            ['full_text', 'created_at', 'username']
        ].head(5).to_string()
    )

else:
    print("\n⚠️ Gagal. Periksa auth token dan koneksi internet.")

🔍 Uji coba scraping 1 keyword...
  ⏭️ [test_tunanetra_mbanking] Sudah ada di folder data, skip scraping.

📋 Preview hasil scraping:
                                                                                                                                                                                                                                                                              full_text                created_at     username
0                                                                                                                 Namun, bagi para tunanetra pengguna Android dan screen reader (pembaca layar) TalkBack, tampilan baru aplikasi BNI Mobile Banking ini menjadi sangat tidak aksesibel.  2021-07-19T03:11:12.000Z     Muha_aku
1                                                                       Contoh nyatanya adalah kendala yang dialami teman-teman tunanetra dalam mengakses aplikasi BNI Mobile Banking versi terbaru di Android. Padahal, jumlah tunane

## Scraping Massal — Layer 1: Aksesibilitas Tunanetra

In [19]:
print("=" * 60)
print("🦯 LAYER 1 — Aksesibilitas Tunanetra")
print("=" * 60)

results_layer1 = {}

for keyword, filename in KEYWORDS_AKSESIBILITAS:
    print(f"\n🔍 Scraping: '{keyword}'")
    ok = scrape_keyword(keyword, filename)
    results_layer1[filename] = ok
    time.sleep(DELAY_SEC)

berhasil = sum(results_layer1.values())
print(f"\n📊 Layer 1 selesai: {berhasil}/{len(KEYWORDS_AKSESIBILITAS)} keyword berhasil")

🦯 LAYER 1 — Aksesibilitas Tunanetra

🔍 Scraping: 'tunanetra BRI'
  ⏭️ [aksesibilitas_bri_tunanetra] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'tunanetra BCA'
  ⏭️ [aksesibilitas_bca_tunanetra] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'tunanetra Mandiri'
  ⏭️ [aksesibilitas_mandiri_tunanetra] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'tunanetra Dana'
  ⏭️ [aksesibilitas_dana_tunanetra] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'tunanetra GoPay'
  ⏭️ [aksesibilitas_gopay_tunanetra] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'tunanetra OVO'
  ⏭️ [aksesibilitas_ovo_tunanetra] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'talkback BRI'
  ⏭️ [talkback_bri] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'talkback BCA'
  ⏭️ [talkback_bca] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'talkback Mandiri'
  ⏭️ [talkback_mandiri] Sudah ada di folder data, skip scraping.

🔍 Scraping: 'talkback Dana'
  ⏭️ [talkback_d

## Gabungkan & Bersihkan Dataset

In [22]:
# =============================================================================
# GABUNGKAN DATASET HASIL SCRAPING
# =============================================================================

all_dfs = []

for keyword, filename in KEYWORDS_AKSESIBILITAS:

    path = f"{OUTPUT_DIR}/{filename}.csv"

    if os.path.exists(path):

        try:
            df_tmp = pd.read_csv(path)

            # Skip jika dataframe kosong
            if df_tmp.empty:
                print(f"⚠️ Skip file kosong: {filename}.csv")
                continue

            # Tambahkan metadata
            df_tmp['keyword'] = keyword
            df_tmp['keyword_file'] = filename
            df_tmp['layer'] = 'Aksesibilitas'

            all_dfs.append(df_tmp)

        except pd.errors.EmptyDataError:
            print(f"⚠️ File benar-benar kosong: {filename}.csv")
            continue

# =============================================================================
# MERGE DAN DEDUPLICATION
# =============================================================================

if not all_dfs:

    print("❌ Tidak ada data yang berhasil dikumpulkan.")

else:

    df_all = pd.concat(all_dfs, ignore_index=True)

    print(f"📦 Total sebelum dedup : {len(df_all):,} tweets")

    # Hapus duplikat berdasarkan isi tweet
    df_all.drop_duplicates(
        subset='full_text',
        inplace=True
    )

    df_all.reset_index(drop=True, inplace=True)

    print(f"✅ Total setelah dedup : {len(df_all):,} tweets")

    # Simpan dataset gabungan
    FINAL_PATH = (
        f"{OUTPUT_DIR}/voicebank_twitter_dataset.csv"
    )

    df_all.to_csv(FINAL_PATH, index=False)

    print(f"\n💾 Dataset tersimpan: {FINAL_PATH}")

⚠️ File benar-benar kosong: talkback_gopay.csv
⚠️ File benar-benar kosong: aksesibilitas_ewallet.csv
⚠️ File benar-benar kosong: disabilitas_visual_bank.csv
⚠️ File benar-benar kosong: tunanetra_bayar.csv
📦 Total sebelum dedup : 1,259 tweets
✅ Total setelah dedup : 738 tweets

💾 Dataset tersimpan: ../data/voicebank_twitter_dataset.csv


## Labeling — Klasifikasi App & Sentimen Awal

In [23]:
# =============================================================================
# LABELING AWAL DATASET
# =============================================================================

# -------------------------------------------------------------------------
# Deteksi nama aplikasi dari isi tweet
# -------------------------------------------------------------------------

APP_MAP = {

    'BRI': ['bri', 'brimo', 'briva'],

    'BCA': ['bca', 'klik bca', 'mybca'],

    'Mandiri': [
        'mandiri',
        'livin',
        'livin mandiri'
    ],

    'Dana': ['dana'],

    'GoPay': ['gopay', 'gojek'],

    'OVO': ['ovo'],
}


def detect_app(text):

    """
    Deteksi aplikasi keuangan
    berdasarkan isi tweet.
    """

    text_lower = str(text).lower()

    detected = []

    for app, keywords in APP_MAP.items():

        if any(kw in text_lower for kw in keywords):
            detected.append(app)

    return (
        ', '.join(detected)
        if detected
        else 'Tidak Terdeteksi'
    )


# -------------------------------------------------------------------------
# Deteksi istilah aksesibilitas eksplisit
# -------------------------------------------------------------------------

AKSESIBILITAS_TERMS = [

    'tunanetra',
    'buta',
    'blind',
    'low vision',
    'gangguan penglihatan',

    'disabilitas visual',
    'difabel',
    'disabilitas',

    'talkback',
    'talk back',
    'screen reader',
    'voiceover',
    'voice over',

    'aksesibilitas',
    'aksesibel',
    'accessibility',
    'accessible',
]


def has_aksesibilitas_term(text):

    """
    Memeriksa apakah tweet mengandung
    istilah aksesibilitas eksplisit.
    """

    text_lower = str(text).lower()

    return any(
        term in text_lower
        for term in AKSESIBILITAS_TERMS
    )


# =============================================================================
# TERAPKAN LABELING
# =============================================================================

df_all['app_detected'] = (
    df_all['full_text']
    .apply(detect_app)
)

df_all['has_aksesibilitas'] = (
    df_all['full_text']
    .apply(has_aksesibilitas_term)
)

# =============================================================================
# RINGKASAN HASIL LABELING
# =============================================================================

print("✅ Labeling selesai!")

print("\n📱 Distribusi aplikasi terdeteksi:")
print(
    df_all['app_detected']
    .value_counts()
    .head(10)
)

print(
    f"\n🦯 Tweet dengan term aksesibilitas eksplisit: "
    f"{df_all['has_aksesibilitas'].sum():,}"
)

# =============================================================================
# UPDATE DATASET FINAL
# =============================================================================

df_all.to_csv(FINAL_PATH, index=False)

print(f"\n💾 Dataset diperbarui: {FINAL_PATH}")

✅ Labeling selesai!

📱 Distribusi aplikasi terdeteksi:
app_detected
Tidak Terdeteksi    246
BCA                 175
Dana                131
Mandiri             104
BRI                  60
BCA, Mandiri          7
OVO                   5
BCA, Dana, GoPay      3
BRI, Mandiri          1
BRI, Dana             1
Name: count, dtype: int64

🦯 Tweet dengan term aksesibilitas eksplisit: 635

💾 Dataset diperbarui: ../data/voicebank_twitter_dataset.csv


In [31]:
# =============================================================================
# FILTER RELEVANSI TWEET
# Menyimpan tweet yang benar-benar relevan dengan:
# 1. Aksesibilitas tunanetra
# 2. Konteks layanan keuangan digital
# =============================================================================

# -------------------------------------------------------------------------
# Kata kunci konteks finansial
# -------------------------------------------------------------------------

FINANCIAL_TERMS = [

    'bank',
    'mobile banking',
    'mbanking',
    'm-banking',

    'atm',
    'transfer',
    'saldo',
    'rekening',

    'brimo',
    'bca',
    'livin',
    'mandiri',

    'dana',
    'gopay',
    'ovo',

    'ewallet',
    'e-wallet',
]

# -------------------------------------------------------------------------
# Kata kunci konteks aksesibilitas
# -------------------------------------------------------------------------

ACCESSIBILITY_TERMS = [

    'tunanetra',
    'blind',
    'buta',
    'low vision',

    'talkback',
    'talk back',

    'screen reader',
    'voiceover',
    'voice over',

    'aksesibilitas',
    'aksesibel',

    'disabilitas visual',
    'difabel',
]

# -------------------------------------------------------------------------
# Blacklist tweet tidak relevan/noise
# -------------------------------------------------------------------------

BLACKLIST_TERMS = [

    'infaq',
    'sedekah',
    'donasi',
    'yatim',

    'sembako',
    'bingkisan',
    'ta’jil',
    "ta'jil",

    'open donasi',
    'bantuan sosial',
]


# =============================================================================
# FUNGSI FILTER RELEVANSI
# =============================================================================

def is_relevant_tweet(text):

    text_lower = str(text).lower()

    # Apakah mengandung konteks finansial
    has_financial = any(
        term in text_lower
        for term in FINANCIAL_TERMS
    )

    # Apakah mengandung konteks aksesibilitas
    has_accessibility = any(
        term in text_lower
        for term in ACCESSIBILITY_TERMS
    )

    # Apakah termasuk noise
    has_blacklist = any(
        term in text_lower
        for term in BLACKLIST_TERMS
    )

    # Relevan jika:
    # financial + accessibility + bukan noise
    return (
        has_financial
        and has_accessibility
        and not has_blacklist
    )


# =============================================================================
# TERAPKAN FILTER
# =============================================================================

before_filter = len(df_all)

df_all = df_all[
    df_all['full_text']
    .apply(is_relevant_tweet)
]

df_all.reset_index(drop=True, inplace=True)

# =============================================================================
# HASIL FILTERING
# =============================================================================

print("✅ Filtering relevansi selesai!")

print(f"\n📦 Sebelum filtering : {before_filter:,} tweets")
print(f"🎯 Setelah filtering : {len(df_all):,} tweets")

print(
    f"🧹 Tweet yang dibuang : "
    f"{before_filter - len(df_all):,} tweets"
)

# =============================================================================
# UPDATE DATASET FINAL
# =============================================================================

df_all.to_csv(FINAL_PATH, index=False)

print(f"\n💾 Dataset diperbarui: {FINAL_PATH}")

✅ Filtering relevansi selesai!

📦 Sebelum filtering : 583 tweets
🎯 Setelah filtering : 531 tweets
🧹 Tweet yang dibuang : 52 tweets

💾 Dataset diperbarui: ../data/voicebank_twitter_dataset.csv


## Eksplorasi Awal Dataset

In [32]:
# =============================================================================
# RINGKASAN DATASET
# =============================================================================

print("=" * 60)
print("📊 RINGKASAN DATASET")
print("=" * 60)

# -------------------------------------------------------------------------
# Informasi umum dataset
# -------------------------------------------------------------------------

print(f"\n📦 Total tweet : {len(df_all):,}")

print("\n📑 Kolom dataset:")
print(list(df_all.columns))

# -------------------------------------------------------------------------
# Distribusi aplikasi yang terdeteksi
# -------------------------------------------------------------------------

print(f"\n{'─' * 40}")
print("📱 Distribusi aplikasi terdeteksi:")

print(
    df_all['app_detected']
    .value_counts()
    .head(10)
)

# -------------------------------------------------------------------------
# Distribusi tweet dengan term aksesibilitas
# -------------------------------------------------------------------------

print(f"\n{'─' * 40}")
print("🦯 Tweet mengandung term aksesibilitas:")

print(
    df_all['has_aksesibilitas']
    .value_counts()
)

# -------------------------------------------------------------------------
# Preview tweet
# -------------------------------------------------------------------------

print(f"\n{'─' * 40}")
print("📝 Preview 5 tweet dataset:")

sample_tweets = df_all[
    ['username', 'full_text', 'app_detected']
].head(5)

for _, row in sample_tweets.iterrows():

    print(f"\n@{row['username']} [{row['app_detected']}]")

    print(
        row['full_text'][:200]
    )

📊 RINGKASAN DATASET

📦 Total tweet : 531

📑 Kolom dataset:
['conversation_id_str', 'created_at', 'favorite_count', 'full_text', 'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location', 'quote_count', 'reply_count', 'retweet_count', 'tweet_url', 'user_id_str', 'username', 'keyword', 'keyword_file', 'layer', 'app_detected', 'has_aksesibilitas']

────────────────────────────────────────
📱 Distribusi aplikasi terdeteksi:
app_detected
BCA                 138
Tidak Terdeteksi    129
Dana                124
Mandiri             100
BRI                  25
BCA, Mandiri          3
BCA, Dana, GoPay      3
OVO                   3
BRI, Mandiri          1
BRI, Dana             1
Name: count, dtype: int64

────────────────────────────────────────
🦯 Tweet mengandung term aksesibilitas:
has_aksesibilitas
True    531
Name: count, dtype: int64

────────────────────────────────────────
📝 Preview 5 tweet dataset:

@RobinnapituS [BRI]
@promo_BRI Tolong tunjukkan undang-undangnya bahwa tunanetra

In [ ]:
# Tampilkan info kolom lengkap
print("\n📋 Struktur DataFrame:")
df_all.info()

print("\n📋 Sample data:")
df_all.head(3)